In [1]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [2]:
from google.colab import files
files.upload()

Saving archive (1).zip to archive (1).zip


In [9]:
import zipfile

with zipfile.ZipFile("archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [10]:
import os
print(os.listdir("dataset"))

['Data']


In [11]:
print(os.listdir("dataset/Data"))

['REAL', 'FAKE']


In [12]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_data = ImageFolder(
    "dataset/Data",
    transform=train_transform
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

In [13]:
images, labels = next(iter(train_loader))
print(images.shape)

torch.Size([32, 3, 224, 224])


In [14]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H/patch, W/patch)
        x = x.flatten(2)  # flatten patches
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

In [ ]:
import os

print("Contents of current directory:")
print(os.listdir('.'))

print("\nContents of 'dataset' directory (if it exists):")
try:
    print(os.listdir("dataset"))
except FileNotFoundError:
    print("'dataset' directory not found.")